In [1]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import json
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from '/home/sagemaker-user/analysis-tools/src/training/gbm_model_trainer.py'>

## Load data

In [2]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight functionality of plot_target_vs_predictors()
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(df["random"]  > 0.70, "V", "T")

In [5]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.698304
V    0.301696
Name: proportion, dtype: float64

In [6]:
df.groupby("split")["target"].mean()

split
T    0.118336
V    0.113856
Name: target, dtype: float64

In [7]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])

## Use ModelTrainer class for training
- Test hyperparameter tuning

In [160]:
# Define your config
config_xgb = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        "objective": "binary:logistic",
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    output_email=True,
    log_file="outputs/training_xgb.log",

    # Reporting parameters
    report_output_path="outputs/model_analysis_xgb.html",
    report_params={
    },
    email="asacco@plymouthrock.com",
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to JSON
config_output_path = Path("model_config_xgb.yaml")

# Write config to JSON file
with config_output_path.open("w") as f:
    json.dump(asdict(config_xgb), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")

✅ Config saved to /home/sagemaker-user/analysis-tools/notebooks/model_config_xgb.yaml


In [161]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)

<module 'analysis.plots' from '/home/sagemaker-user/analysis-tools/src/analysis/plots.py'>

In [162]:
from training.gbm_model_trainer import GBMModelTrainer
from xgboost import XGBClassifier

mt_xgboost = GBMModelTrainer(
    model_class=XGBClassifier,
    config_path="model_config_xgb.yaml",
    train_df=train,
    valid_df=test
)

In [163]:
mt_xgboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', log_file='outputs/training_xgb.log', report_output_path='outputs/model_analysis_xgb.html', email='asacco@plymouthrock.com', hyperparameters={'objective': 'binary:logistic', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}, output_log=True, output_email=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type': 'no

In [164]:
mt_xgboost.train()

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/xgboost/core.py:160: UserWarning: [15:34:10] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "early_stopping" } are not used.

  warnings.warn(smsg, UserWarning)
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_

✅ Analysis report generated at outputs/model_analysis_xgb.html


## Test with CAT Boost

In [165]:
from training.gbm_model_trainer import GBMModelTrainer
from catboost import CatBoostClassifier

In [166]:
# Define your config
config_catboost = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        # "objective": "binary:logistic",
        "loss_function": "Logloss", # Valid for CATBoost
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    output_email=True,
    email="asacco@plymouthrock.com",
    log_file="outputs/training_catboost.log",

    # Reporting parameters
    report_output_path="outputs/model_analysis_catboost.html",
    report_params={
    },
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to JSON
config_output_path = Path("model_config_catboost.yaml")

# Write config to JSON file
with config_output_path.open("w") as f:
    json.dump(asdict(config_catboost), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")

✅ Config saved to /home/sagemaker-user/analysis-tools/notebooks/model_config_catboost.yaml


In [167]:
mt_catboost = GBMModelTrainer(
    model_class=CatBoostClassifier,
    config_path="model_config_catboost.yaml",
    train_df=train,
    valid_df=test
)

In [168]:
mt_catboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', log_file='outputs/training_catboost.log', report_output_path='outputs/model_analysis_catboost.html', email='asacco@plymouthrock.com', hyperparameters={'loss_function': 'Logloss', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}, output_log=True, output_email=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type

In [169]:
mt_catboost.train()

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-

✅ Analysis report generated at outputs/model_analysis_catboost.html


## Test with LightGBM

In [170]:
# Define your config
config_lgbm = GBMModelTrainerConfig(

    # Training parameters
    actual_col="target",
    predicted_col="pred_xgb",

    hyperparameters={
        # "objective": "binary:logistic",
        "loss_function": "Logloss", # Valid for LightGBM
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "colsample_bytree": 0.5,
        "random_state": 42,
        "early_stopping": 25,
    },

    # Logging parameters
    output_log=True,
    output_report=True,
    output_email=True,
    email="asacco@plymouthrock.com",
    log_file="outputs/training_lgbm.log",

    # Reporting parameters
    report_output_path="outputs/model_analysis_catboost.html",
    report_params={
    },
    tabulate_vars=["job", "marital", "education"],
    plots_to_add=[
        {
            "plot": "plot_error_by_group_grid",
            "title": "Error by Analysis Variables",
            "kwargs": {
                "group_cols": ["job", "education", "age"]
            }
        },
        {
            "plot": "gain_curve_with_gini",
            "title": "Gain Curve / Lorenz Curve"
        },
        {
            "plot": "partial_gini_plot",
            "title": "Partial Gini (Top 15%)",
            "kwargs": {
                "top_percent": 15
            }
        },
        {
            "plot": "lift_chart",
            "title": "Lift Chart"
        },
        {
            "plot": "crunched_residual_plot",
            "title": "Crunched Residuals"
        },
        {
            "plot": "plot_residual_fit",
            "title": "Std and Avg of Normalized Residuals",
            "kwargs": {
                "residual_type": "normalized"
            }
        }
    ],
)

# Output config to YAML
config_output_path = Path("model_config_lgbm.yaml")

# Write config to YAML file
with config_output_path.open("w") as f:
    json.dump(asdict(config_lgbm), f, indent=2)

print(f"✅ Config saved to {config_output_path.resolve()}")

✅ Config saved to /home/sagemaker-user/analysis-tools/notebooks/model_config_lgbm.yaml


In [171]:
mt_lgbm = GBMModelTrainer(
    model_class=CatBoostClassifier,
    config_path="model_config_lgbm.yaml",
    train_df=train,
    valid_df=test
)

In [172]:
mt_lgbm.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred_xgb', log_file='outputs/training_lgbm.log', report_output_path='outputs/model_analysis_catboost.html', email='asacco@plymouthrock.com', hyperparameters={'loss_function': 'Logloss', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}, output_log=True, output_email=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Normalized Residuals', 'kwargs': {'residual_type': '

In [173]:
mt_lgbm.train()

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/seaborn/_oldcore.py:1108: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context("mode.use_inf_as_na", True):
/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-

✅ Analysis report generated at outputs/model_analysis_catboost.html


In [174]:
mt_lgbm.log_lines

["[2025-07-28 15:34:19] ⚠️ 'colsample_bytree' is not a valid hyperparameter for CatBoostClassifier and will be ignored. ",
 "[2025-07-28 15:34:19] ⚠️ 'early_stopping' is not a valid hyperparameter for CatBoostClassifier and will be ignored. ",
 '[2025-07-28 15:34:19] Instantiated model: CatBoostClassifier',
 '[2025-07-28 15:34:20] Training completed in 0.3665 seconds',
 '[2025-07-28 15:34:20] Model: CatBoostClassifier',
 "[2025-07-28 15:34:20] Hyperparameters: {'loss_function': 'Logloss', 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.5, 'colsample_bytree': 0.5, 'random_state': 42, 'early_stopping': 25}",
 '[2025-07-28 15:34:20] Training data shape: (31571, 19)',
 '[2025-07-28 15:34:20] Validation data shape: (13640, 19)',
 '[2025-07-28 15:34:20] Number of predictors: 18',
 "[2025-07-28 15:34:20] Target column: 'target'",
 "[2025-07-28 15:34:20] Prediction column: 'pred_xgb'",
 '[2025-07-28 15:34:20] Validation score (default metric): 0.9081',
 '[2025-07-28 1